# Apex Task 2: run the ROS 2 race

This notebook lists every command for Task 2, in order, with the time each step took on my laptop
(Ubuntu 24.04, 12 cores, CPU rendering in Docker).

There are two ways to use it.

* **Headless check.** Run the code cells from top to bottom. They build the image, start the container,
  run one full mission with scripted key presses, and draw the figures. No window opens.
* **Recorded run with the GUI.** Follow the markdown steps in section 4. They need three terminals,
  because Gazebo and the keyboard node need a real screen and a real keyboard.

All paths are relative to the repo root. The first code cell moves there.

In [ ]:
%cd ..
!pwd

## 1. Build the Docker image (once)

Time: about 3 min 45 s (225 s measured with --no-cache) the first time, with `osrf/ros:humble-desktop-full` already pulled.
Pulling that base image first adds a few minutes more (about 5.4 GB on disk).

In [ ]:
!docker pull osrf/ros:humble-desktop-full
!docker build -t apex:humble task2_ros2/docker

## 2. Start the container and build the package

Time: container start under 1 s (0.14 s), `colcon build` about 2 s (1.7 s).

`run.sh --headless` starts the container in the background with no display.
For the GUI run in section 4 use `task2_ros2/docker/run.sh` with no flag instead.
If a container named `apex` already exists, remove it first with `docker rm -f apex`.

In [ ]:
!docker rm -f apex 2>/dev/null; task2_ros2/docker/run.sh --headless
!docker exec apex bash -c 'source /opt/apex_env.sh && cd /ws && colcon build --symlink-install --base-paths src/apex/task2_ros2'

## 3. Unit tests and world generation

Time: tests under 1 s (42 tests), world generation under 1 s (0.09 s).

The world file is already in the repo. Regenerate it after you change `config/circuit.yaml`.

In [ ]:
!docker exec apex bash -c 'source /opt/apex_env.sh && cd /ws/src/apex/task2_ros2/apex_race && python3 -m pytest -q test'
!docker exec apex bash -c 'cd /ws/src/apex/task2_ros2 && python3 world_gen/generate_world.py'

## 4. Headless mission (scripted keys)

Time: about 4 min 42 s for the whole run. The race itself is 3 laps.

This starts Gazebo with no window, presses g, drops the cone on lap 2,
pauses and resumes once on lap 3, and prints the summary. Logs go to `task2_ros2/runs/<timestamp>/`.

In [ ]:
!docker exec apex bash -c 'source /opt/apex_env.sh && cd /ws/src/apex/task2_ros2 && scripts/headless_run.sh'

Phase 3 version, with the blue player car. The script parks the player car on the front straight before it presses g,
so the racer has to pass it on every lap. Time: about 5 min 27 s.

In [ ]:
!docker exec apex bash -c 'source /opt/apex_env.sh && cd /ws/src/apex/task2_ros2 && scripts/headless_run.sh player'

## 5. Figures from the newest run

Time: about 1 s (1.16 s).

In [ ]:
!docker exec apex bash -c 'cd /ws/src/apex/task2_ros2 && python3 scripts/analyze_run.py $(ls -d runs/*/ | tail -1)'
!ls task2_ros2/runs/$(ls task2_ros2/runs | tail -1)/figures

## 6. 2D LiDAR map of the circuit

Time: about 4 min 38 s (277.5 s, one car) for the run, and about 2 s for the figures.

`slam` adds slam_toolbox to the launch. It builds a 2D map from the racer's LiDAR while the racer drives
the same mission. At the end the script saves the map as `map.pgm` and `map.yaml` in the run folder.
`plot_map.py` draws the map and checks it against the wall positions in `circuit.yaml`.
Add `player` before `slam` to map the two car mission.

In [ ]:
!docker exec apex bash -c 'source /opt/apex_env.sh && cd /ws/src/apex/task2_ros2 && scripts/headless_run.sh slam'
!docker exec apex bash -c 'cd /ws/src/apex/task2_ros2 && python3 scripts/plot_map.py $(ls -d runs/*/ | tail -1)'

## 7. Recorded run with the GUI and your own key presses

Do this in three terminals, not in this notebook. Stop the headless container first:
`docker rm -f apex`.

**Terminal 1** (time to Gazebo window: about 2.2 s to a ready racer with no window; I did not time the Gazebo window itself):
```
cd ~/robotics\ coursework
task2_ros2/docker/run.sh
cd /ws && colcon build --symlink-install --base-paths src/apex/task2_ros2 && source install/setup.bash
cd /ws/src/apex/task2_ros2
ros2 launch apex_race race.launch.py player:=true
```
Leave out `player:=true` for the one car version. Add `slam:=true` to build the 2D map as well.

**Terminal 2**, the keyboard:
```
task2_ros2/docker/exec.sh
ros2 run apex_race keyboard_node --ros-args --params-file apex_race/config/params.yaml
```
Click into this terminal so it gets your keys.

**Terminal 3**, the status:
```
task2_ros2/docker/exec.sh
ros2 topic echo /apex/status
```

**The mission** (about 4.5 to 5 min (267 s one car, 298 s two cars, pause not counted) of race time):
1. Press `g`. The countdown runs 3, 2, 1, Go.
2. Drive the blue car with `w a s d` (space stops it). Park it on the racing line if you want the racer to pass it.
3. On lap 2 the cone drops on the back straight. Watch AVOID and REPLAN in terminal 1.
4. On lap 3 press `p` to pause. Wait a few seconds. Press `p` again to resume.
5. After lap 3 the racer stops and terminal 1 prints the summary.
6. Press `Ctrl+C` in terminal 1, then `q` in terminal 2. The Gazebo window takes about 5 s to close: it ignores the first signal, and the launch then stops it.

Other keys: `x` aborts the race, `r` resets it after it ends or while paused.

If you started with `slam:=true`, save the map before step 6, from terminal 3 (Ctrl+C the echo first):
```
ros2 run nav2_map_server map_saver_cli -f runs/<timestamp>/map --ros-args -p use_sim_time:=true
```

Then draw the figures for that run:
```
python3 scripts/analyze_run.py runs/<timestamp>
python3 scripts/plot_map.py runs/<timestamp>     # only if you saved a map
```

## 8. Clean up

```
docker rm -f apex
```
The image stays. Remove it with `docker rmi apex:humble` if you need the space.